### Saving and loading a scikit-learn model

In [2]:
import sklearn
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn import datasets

# SCikit-learn version
scikit_learn_version = sklearn.__version__

# Load the iris dataset
iris = datasets.load_iris()
features = iris.data
target = iris.target

# Creat classifier
classifier = RandomForestClassifier()

# Train the model
model = classifier.fit(features, target)

# Save the model as a pickle file
joblib.dump(model, "model_{version}.pkl".format(version=scikit_learn_version))

# Load the model from the pickle file
classifier = joblib.load("model_{version}.pkl".format(version=scikit_learn_version))

# Create a new observation
new_observation = [[5.2, 3.2, 1.1, 0.1]]

# Predict the class of the new observation
classifier.predict(new_observation)

array([0])

### Saving and loading a TensorFlow model

In [3]:
import numpy as np
from tensorflow import keras

# Set random seed for reproducibility
np.random.seed(0)

# Create a model with one hidden layer
input_layer = keras.Input(shape=(10,))
hidden_layer = keras.layers.Dense(10)(input_layer)
output_layer = keras.layers.Dense(1)(hidden_layer)
model = keras.Model(input_layer, output_layer)
model.compile(optimizer="adam", loss="mean_squared_error")

# Train the model
x_train = np.random.rand(1000, 10)
y_train = np.random.rand(1000, 1)
model.fit(x_train, y_train, epochs=5, batch_size=32)

# Save the model in Keras format
model.save("saved_model.keras")

# Load the neural network model
loaded_model = keras.models.load_model("saved_model.keras")

# Verify that the loaded model can make predictions
loaded_model.predict(x_train[:1])

I0000 00:00:1787323402.802123  200182 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787323402.902023  200182 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787323404.252188  200182 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787323404.252792  200182 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will 

Epoch 1/5


E0000 00:00:1787323404.474707  200182 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2256  
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 864us/step - loss: 0.1498
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 996us/step - loss: 0.1332
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 926us/step - loss: 0.1205
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 934us/step - loss: 0.1112
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step


array([[0.6928758]], dtype=float32)

### Saving and loading a PyTorch model

In [4]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import RMSprop
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Create a synthetic dataset for binary classification
features, target = make_classification(n_classes=2, n_features=10,
    n_samples=1000)
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.1, random_state=1)

# Set random seed for reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Transform the data into PyTorch tensors
x_train = torch.from_numpy(features_train).float()
y_train = torch.from_numpy(target_train).float().view(-1, 1)
x_test = torch.from_numpy(features_test).float()
y_test = torch.from_numpy(target_test).float().view(-1, 1)

# Define a simple neural network using 'Sequential'
class SimpleNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.sequential = torch.nn.Sequential(
            torch.nn.Linear(10, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 1),
            torch.nn.Dropout(0.1),  # Remove 10% of neurons
            torch.nn.Sigmoid(),
        )

    def forward(self, x):
        return self.sequential(x)


# Initialize the neural network
network = SimpleNeuralNet()

# Define the loss criterion and optimizer
criterion = nn.BCELoss()
optimizer = RMSprop(network.parameters())

# Define the data loader
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=100, shuffle=True)

# Compile the model using the torch 2.0 optimizer
compiled_network = torch.compile(network)

# Train the neural network using backpropagation
epochs = 5
for epoch in range(epochs):
    for data, target in train_loader:
        optimizer.zero_grad()
        output = compiled_network(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    # Save one checkpoint after all batches in the epoch are complete.
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": network.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": loss.item(),
    }
    torch.save(checkpoint, "model.pt")
    print("Epoch:", epoch + 1, "\tloss:", loss.item())

# Initialize the neural network and optimizer for loading
loaded_network = SimpleNeuralNet()
loaded_optimizer = RMSprop(loaded_network.parameters())

checkpoint = torch.load("model.pt", map_location=torch.device("cpu"))
loaded_network.load_state_dict(checkpoint["model_state_dict"])
loaded_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
loaded_network.eval()

Epoch: 1 	loss: 0.13303154706954956
Epoch: 2 	loss: 0.15853147208690643
Epoch: 3 	loss: 0.09414386749267578
Epoch: 4 	loss: 0.1267329305410385
Epoch: 5 	loss: 0.11501558125019073


SimpleNeuralNet(
  (sequential): Sequential(
    (0): Linear(in_features=10, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
    (5): Dropout(p=0.1, inplace=False)
    (6): Sigmoid()
  )
)

### Deploying scikit-learn models

In [9]:
import joblib
from flask import Flask, request

# Initialize the Flask application
app = Flask(__name__)

# Load the model from disk
model = joblib.load("model_{version}.pkl".format(version=scikit_learn_version))

# Create a route for the prediction endpoint
@app.route("/predict", methods = ["POST"])
def predict():
    print(request.json)
    inputs = request.json["inputs"]
    prediction = model.predict(inputs)
    return {
        "prediction": prediction.tolist()
    }
    
# Run the Flask application
if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [21/Aug/2026 17:50:11] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [21/Aug/2026 17:50:11] "GET /favicon.ico HTTP/1.1" 404 -
